# 13D — Upstream precipitation and river-stage evidence

**Priority 4.** Match independently obtained gauge observations to actual SAR acquisition times; optional upstream basin rainfall. The package does **not** contain real gauge or basin datasets and does not invent them.

**Execution:** This is a source workflow, NOT results. A successful extract needs your own data / Earth Engine session. Do not treat low-VV detections as validated water, flood depth or methane flux.

In [1]:
# Every notebook uses the same ROOT; no hard-coded drive or EE project ID.
from pathlib import Path
import os, sys
starting = Path(os.environ.get("HAKALUKI_ROOT", Path.cwd())).expanduser().resolve()
for candidate in [starting, *starting.parents]:
    if (candidate/"src/analysis13/core.py").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Extract the ZIP into the project root and open Jupyter there, or set HAKALUKI_ROOT")
sys.path.insert(0, str(ROOT))
print("Research root:", ROOT)


Research root: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project


## Review actual-data input requirements

Real files must be saved under `data/external/hydrology`. Templates are empty headers and must be replaced with source-documented measurements. Stage datum and hydraulic representativeness matter.

In [2]:
from src.analysis13.hydrology import stage_input, basin_rain_input
print('Gauge input:',stage_input(ROOT))
print('Upstream rainfall input:',basin_rain_input(ROOT))
print('Do not use CHIRPS at the haor as a substitute for transboundary inflow or gauge stage.')

Gauge input: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\data\external\hydrology\observed_stage.csv
Upstream rainfall input: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\data\external\hydrology\upstream_daily_rain.csv
Do not use CHIRPS at the haor as a substitute for transboundary inflow or gauge stage.


## Compute chronology and conditionally match stage data

Generates the SAR chronology even if gauge files are missing, recording explicit acquisition/data gaps.

In [3]:
from src.analysis13.hydrology import run_hydrology
status=run_hydrology(ROOT)
print(status)

{'created_utc': '2026-10-14T17:27:45.893905+00:00', 'status': 'scene_chronology_completed_external_hydrology_pending', 'screening_cutoff_fraction': 0.85, 'reason': 'An arbitrary screening criterion is not a true flood-onset rule.', 'stage_file': None, 'upstream_file': None, 'stage_blocker': 'No actual station data at F:\\Wetland Test\\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\\data\\external\\hydrology\\observed_stage.csv. Obtain source/datum/station metadata.', 'upstream_blocker': 'No measured/upstream basin rain data at F:\\Wetland Test\\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\\data\\external\\hydrology\\upstream_daily_rain.csv. Do not substitute local CHIRPS.'}


## Inspect matching support and stage series

No stage–area calibration is attempted without suitable independent water-area classification and hydrologically relevant gauge observations.

In [4]:
from pathlib import Path
for p in sorted((ROOT/'results/tables/analysis13').glob('13D_*')):
    print(p.name, p.stat().st_size, 'bytes')

13D_first_observed_screening_crossings.csv 1030 bytes
13D_hydrology_data_availability.json 713 bytes
13D_observed_within_orbit_windows.csv 29663 bytes
